# 01_dataset — Peek inside a LeRobotDataset

Load a `LeRobotDataset` (v3.0 format, lerobot 0.5.1), inspect its metadata and tensor
shapes, take a temporal window, run one DataLoader batch, and plot a camera frame and
an episode's action/state.

**Where the data comes from.** This notebook reads `DATA_REPO` (set via `config.env`).
For an offline/standalone run, point it at the synthetic dataset built by the tools:

```bash
python tools/make_synthetic_dataset.py --format lerobot --root .smoke/synthetic
export DATA_REPO=handson/synthetic
export LEROBOT_DATASET_ROOT=$PWD/.smoke/synthetic   # optional: local dir for the dataset
```

On Miyabi, `DATA_REPO` is a real repo pre-downloaded into `HF_HOME` and you can omit
`LEROBOT_DATASET_ROOT`.

In [ ]:
import os

REPO_ID = os.environ.get("DATA_REPO", "handson/synthetic")
DATASET_ROOT = os.environ.get("LEROBOT_DATASET_ROOT") or None  # local dir, or None for HF cache
assert not REPO_ID.startswith("<TODO"), "Run `source config.env` or set DATA_REPO."
# On an offline compute node, only the pre-downloaded cache is used:
# os.environ["HF_HUB_OFFLINE"] = "1"
print("DATA_REPO            =", REPO_ID)
print("LEROBOT_DATASET_ROOT =", DATASET_ROOT)

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

dataset = LeRobotDataset(REPO_ID, root=DATASET_ROOT)
print("fps          =", dataset.meta.fps)
print("num_episodes =", dataset.num_episodes)
print("num_frames   =", dataset.num_frames)

## Metadata: features, stats, tasks

`dataset.meta` holds the schema (feature names, dtypes, shapes), normalization stats,
and the natural-language tasks.

In [ ]:
print("=== features ===")
for name, spec in dataset.meta.features.items():
    print(f"{name:32s} dtype={spec['dtype']:8s} shape={tuple(spec['shape'])}")

print("\n=== tasks ===")
print(dataset.meta.tasks)

print("\n=== stats (action) ===")
act_stats = dataset.meta.stats.get("action", {})
for k, v in act_stats.items():
    try:
        print(f"action.{k}: shape {tuple(v.shape)}")
    except AttributeError:
        print(f"action.{k}: {v}")

In [ ]:
# One frame: what keys and tensor shapes exist?
sample = dataset[0]
for key, value in sample.items():
    shape = tuple(value.shape) if hasattr(value, "shape") else value
    print(f"{key:32s} {shape}")

print("\naction shape =", tuple(sample["action"].shape))
print("state  shape =", tuple(sample["observation.state"].shape))

## Temporal windows with delta_timestamps

`delta_timestamps` (seconds relative to the current frame t) stacks neighboring frames
onto a new leading time axis `T`. This is how ACT / Diffusion Policy consume multiple
steps of observation/action.

In [ ]:
dt = 1.0 / dataset.meta.fps
delta_timestamps = {"observation.state": [-2 * dt, -dt, 0.0]}  # two past frames + current
ds_t = LeRobotDataset(REPO_ID, root=DATASET_ROOT, delta_timestamps=delta_timestamps)
s = ds_t[10]
print("state shape without window :", tuple(sample["observation.state"].shape))
print("state shape with    window :", tuple(s["observation.state"].shape), "  # leading T=3")

## One DataLoader batch

In [ ]:
import torch

loader = torch.utils.data.DataLoader(dataset, batch_size=4, shuffle=True, num_workers=0)
batch = next(iter(loader))
print("batch action shape :", tuple(batch["action"].shape))
print("batch image  shape :", tuple(batch["observation.images.front"].shape))

## Visualize: a camera frame and one episode's signals

In [ ]:
import matplotlib.pyplot as plt

img = sample["observation.images.front"]  # (C, H, W), float in [0, 1]
plt.figure(figsize=(3, 3))
plt.imshow(img.permute(1, 2, 0).cpu().numpy())
plt.title("observation.images.front")
plt.axis("off")
plt.show()

In [ ]:
# Collect one full episode (episode 0) and plot its action/state channels over time.
# Episode boundaries live in dataset.meta.episodes (global frame indices).
ep0 = dataset.meta.episodes[0]
from_idx, to_idx = ep0["dataset_from_index"], ep0["dataset_to_index"]
actions = torch.stack([dataset[i]["action"] for i in range(from_idx, to_idx)]).numpy()
states = torch.stack([dataset[i]["observation.state"] for i in range(from_idx, to_idx)]).numpy()

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(actions)
ax[0].set_title("episode 0 — action channels")
ax[0].set_xlabel("frame")
ax[1].plot(states)
ax[1].set_title("episode 0 — state channels")
ax[1].set_xlabel("frame")
plt.tight_layout()
plt.show()
print("episode 0 length:", to_idx - from_idx, "frames")

## 🔧 Try it

1. **Predict then check.** Change `delta_timestamps` to `[-4*dt, -2*dt, 0.0]`. Before
   running, predict the new `observation.state` shape, then verify.
2. **Mean action.** Compute the mean `action` over episode 0 (use the `actions` array
   above). Which channel has the largest magnitude?
3. **Add images to the window.** Add `"observation.images.front"` to `delta_timestamps`
   and confirm the image tensor gains a leading `T` (shape becomes `(T, C, H, W)`).